<br>
<a href="https://www.nvidia.com/en-us/training/">
    <div style="width: 55%%; background-color: white; margin-top: 50px;">
    <img src="https://dli-lms.s3.amazonaws.com/assets/general/nvidia-logo.png"
         width="400"
         height="186"
         style="margin: 0px -25px -5px; width: 300px"/></div>
</a>
<h1 style="line-height: 1.4;"><font color="#76b900"><b>Building RAG Agents with LLMs</b></font></h1>
<h2><b>Notebook 3: </b>LangChain Expression Language</h2>
<br>

이전 노트북에서는 LLM 애플리케이션에 활용할 몇 가지 서비스, 즉 외부 LLM 플랫폼과 로컬에서 호스팅되는 프론트엔드 서비스를 소개했습니다. 두 구성 요소 모두 LangChain을 포함하지만, 아직 LangChain 자체를 주요하게 다루지는 않았습니다. LangChain과 LLM에 어느 정도 경험이 있다고 가정하지만, 이 노트북은 이후 섹션을 준비할 수 있도록 내용을 따라잡는 것을 목적으로 합니다!

이 노트북은 대규모 언어 모델(LLM)을 위한 대표적인 오케스트레이션 라이브러리인 LangChain을 지난 시간의 AI Foundation Endpoints와 통합하고 활용하는 과정을 안내합니다. 숙련된 개발자든 LLM이 처음이든, 이 코스는 정교한 LLM 애플리케이션을 구축하는 이해와 역량을 높여 줄 것입니다.

<br>

### **학습 목표:**

- 체인(chain)과 러너블(runnable)을 활용해 흥미로운 LLM 시스템을 오케스트레이션하는 방법을 배웁니다.  
- 외부 대화와 내부 추론에 LLM을 사용하는 데 익숙해집니다.
- 노트북 안에서 간단한 [**Gradio**](https://www.gradio.app/) 인터페이스를 띄우고 실행할 수 있게 됩니다.

<br>

### **생각해 볼 질문:**

- 파이프라인을 통해 정보가 계속 흐르도록 하려면 어떤 종류의 유틸리티가 필요할까요 **(다음 노트북의 예고편)**.
- [**Gradio**](https://www.gradio.app/)를 만나면, 이런 스타일의 인터페이스를 어디서 본 적이 있는지 떠올려 보세요. [**HuggingFace Spaces**](https://huggingface.co/spaces) 같은 곳일 수 있습니다...
- 섹션 끝부분에서 체인을 라우트로 전달하고 포트를 통해 여러 환경에서 접근할 수 있다는 것을 배우게 됩니다. 다른 마이크로서비스로부터 체인을 받으려 한다면 어떤 요구 사항을 명시해야 할까요?

<br>

### **환경 설정:**

In [ ]:
## Necessary for Colab, not necessary for course environment
# %pip install -q langchain langchain-nvidia-ai-endpoints gradio

# import os
# os.environ["NVIDIA_API_KEY"] = "nvapi-..."

## If you encounter a typing-extensions issue, restart your runtime and try again
# from langchain_nvidia_ai_endpoints import ChatNVIDIA
# ChatNVIDIA.get_available_models()

<br>

### **모델 고려하기**

코스 카탈로그는 의도적으로 작게 유지됩니다. `nvidia/nemotron-3.5-lightning-30b-a3b`가 텍스트 채팅과 지시 작업을 담당하며, 수업에서 명시적으로 활성화하지 않는 한 thinking은 비활성화됩니다. `meta/llama-3.2-11b-vision-instruct`는 이미지 입력이 필요한 인터페이스를 위해 남겨 두었습니다. 이렇게 하면 각 엔드포인트가 주변 예제와 잘 맞아떨어집니다.

이전 노트북의 `scope=all` 요청을 통해 더 넓은 라이브 카탈로그를 여전히 확인할 수 있습니다. 모델을 교체하기 전에 해당 모델이 체인에 필요한 인터페이스와 응답 동작을 지원하는지 확인하세요.

----

<br>

## **Part 1:** LangChain이란?

LangChain은 하나 이상의 LLM 구성 요소를 가진 시스템을 구축하는 데 도움을 주는 인기 있는 LLM 오케스트레이션 라이브러리입니다. 좋든 나쁘든 이 라이브러리는 매우 인기가 많고 분야의 새로운 발전에 따라 빠르게 변하기 때문에, 누군가는 LangChain의 어떤 부분에는 경험이 많으면서도 다른 부분에는 거의 익숙하지 않을 수 있습니다(기능이 워낙 많거나, 해당 영역이 새로워서 기능이 최근에야 구현되었기 때문입니다).

이 노트북은 **LangChain Expression Language(LCEL)** 를 사용해 기본적인 체인 명세에서 더 고급 대화 관리 방식으로 단계적으로 나아갑니다. 즐거운 여정이 되길 바라며, 숙련된 LangChain 개발자도 새로운 것을 배울 수 있기를 바랍니다!

<!-- > <img style="max-width: 400px;" src="imgs/langchain-diagram.png" /> -->
> <img src="https://dli-lms.s3.amazonaws.com/assets/s-fx-15-v1/imgs/langchain-diagram.png" width=400px/>
<!-- > <img src="https://drive.google.com/uc?export=view&id=1NS7dmLf5ql04o5CyPZnd1gnXXgO8-jbR" width=400px/> -->

----

<br>

## **Part 2:** Chain과 Runnable

새로운 라이브러리를 탐색할 때는 그 라이브러리의 핵심 시스템이 무엇이고 어떻게 사용되는지 파악하는 것이 중요합니다.

LangChain에서 주요 빌딩 블록은 *과거에는* 고전적인 **Chain**이었습니다. 특정한 일을 수행하는 작은 기능 모듈로, 다른 체인과 연결해 시스템을 만들 수 있습니다. 사실상 "빌딩 블록 시스템" 추상화로, 블록을 만들기 쉽고, 일관된 메서드(`invoke`, `generate`, `stream` 등)를 가지며, 서로 연결해 하나의 시스템으로 동작할 수 있습니다. 레거시 체인의 예로는 `LLMChain`, `ConversationChain`, `TransformationChain`, `SequentialChain` 등이 있습니다.

최근에는 훨씬 다루기 쉽고 매우 간결한 새로운 권장 명세인 **LangChain Expression Language(LCEL)** 가 등장했습니다. 이 새로운 형식은 다른 종류의 기본 요소인 **Runnable**에 의존하는데, 이는 단순히 함수를 감싼 객체입니다. 딕셔너리가 암묵적으로 Runnable로 변환되게 하고, **파이프 |** 연산자가 왼쪽에서 오른쪽으로 데이터를 전달하는 Runnable을 만들게 하면(즉, `fn1 | fn2`는 Runnable), 복잡한 로직을 간단하게 명세할 수 있습니다!

다음은 `RunnableLambda` 클래스로 만든 매우 대표적인 Runnable 예시입니다:

In [ ]:
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from functools import partial

################################################################################
## Very simple "take input and return it"
identity = RunnableLambda(lambda x: x)  ## Or RunnablePassthrough works

################################################################################
## Given an arbitrary function, you can make a runnable with it
def print_and_return(x, preface=""):
    print(f"{preface}{x}")
    return x

rprint0 = RunnableLambda(print_and_return)

################################################################################
## You can also pre-fill some of values using functools.partial
rprint1 = RunnableLambda(partial(print_and_return, preface="1: "))

################################################################################
## And you can use the same idea to make your own custom Runnable generator
def RPrint(preface=""):
    return RunnableLambda(partial(print_and_return, preface=preface))

################################################################################
## Chaining two runnables
chain1 = identity | rprint0
chain1.invoke("Hello World!")
print()

################################################################################
## Chaining that one in as well
output = (
    chain1           ## Prints "Welcome Home!" & passes "Welcome Home!" onward
    | rprint1        ## Prints "1: Welcome Home!" & passes "Welcome Home!" onward
    | RPrint("2: ")  ## Prints "2: Welcome Home!" & passes "Welcome Home!" onward
).invoke("Welcome Home!")

## Final Output Is Preserved As "Welcome Home!"
print("\nOutput:", output)

----

<br>

## **Part 3:** 채팅 모델과 딕셔너리 파이프라인

Runnable로 할 수 있는 일은 많지만, 몇 가지 모범 사례를 정립하는 것이 중요합니다. 현재로서는 몇 가지 핵심적인 이유로 *딕셔너리*를 기본 변수 컨테이너로 사용하는 것이 가장 쉽습니다:

**딕셔너리를 전달하면 변수를 이름으로 추적할 수 있습니다.**

딕셔너리는 이름이 있는 변수(키로 참조되는 값)를 전파할 수 있게 해 주므로, 체인 구성 요소의 출력과 기대 입력을 고정하는 데 아주 좋습니다.

**LangChain 프롬프트는 값들의 딕셔너리를 기대합니다.**

LCEL에서 딕셔너리를 받아 문자열을 만드는 LLM 체인을 명세하는 것은 매우 직관적이며, 그 문자열을 다시 딕셔너리로 끌어올리는 것도 마찬가지로 쉽습니다. 이는 매우 의도적인 설계이며, 부분적으로는 위의 이유 때문입니다. 

<br>

### **예제 1:** 간단한 LLM Chain

고전적인 LangChain의 가장 기본적인 구성 요소 중 하나는 **프롬프트**와 **LLM**을 받는 `LLMChain`입니다:

- 프롬프트는 보통 `PromptTemplate.from_template("string with {key1} and {key2}")` 같은 호출로 얻으며, 문자열을 출력으로 만들기 위한 템플릿을 지정합니다. 딕셔너리 `{"key1" : 1, "key2" : 2}`를 넘기면 출력 `"string with 1 and 2"`를 얻을 수 있습니다.
    - `ChatNVIDIA` 같은 채팅 모델에서는 대신 `ChatPromptTemplate.from_messages`를 사용합니다.
- LLM은 문자열을 받아 생성된 문자열을 반환합니다.
    - `ChatNVIDIA` 같은 채팅 모델은 대신 메시지로 동작하지만, 같은 아이디어입니다! 끝에 **StrOutputParser**를 사용하면 메시지에서 내용을 추출할 수 있습니다.

다음은 위에서 설명한 간단한 채팅 체인의 가벼운 예시입니다. 입력 딕셔너리를 받아 전체 메타 목표를 지정하는 시스템 메시지와 모델에 질의할 사용자 입력을 채우는 것이 전부입니다.

In [ ]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

## Simple Chat Pipeline
chat_llm = ChatNVIDIA(model="nvidia/nemotron-3.5-lightning-30b-a3b", timeout=300, model_kwargs={"chat_template_kwargs": {"enable_thinking": False}})

prompt = ChatPromptTemplate.from_messages([
    ("system", "Only respond in rhymes"),
    ("user", "{input}")
])

rhyme_chain = prompt | chat_llm | StrOutputParser()

print(rhyme_chain.invoke({"input" : "Tell me about birds!"}))

<br>

코드 명령을 그대로 사용하는 것 외에도, [**Gradio 인터페이스**](https://www.gradio.app/guides/creating-a-chatbot-fast)를 사용해 모델을 가지고 놀아볼 수 있습니다. Gradio는 커스텀 생성형 AI 인터페이스를 만들기 위한 간단한 빌딩 블록을 제공하는 인기 도구입니다! 아래 예시는 이 예제 체인으로 손쉽게 Gradio 채팅 인터페이스를 만드는 방법을 보여 줍니다:

In [ ]:
import gradio as gr

#######################################################
## Non-streaming Interface like that shown above

# def rhyme_chat(message, history):
#     return rhyme_chain.invoke({"input" : message})

# gr.ChatInterface(rhyme_chat).launch()

#######################################################
## Streaming Interface

def rhyme_chat_stream(message, history):
    ## This is a generator function, where each call will yield the next entry
    buffer = ""
    for token in rhyme_chain.stream({"input" : message}):
        buffer += token
        yield buffer

## Uncomment when you're ready to try this.
demo = gr.ChatInterface(rhyme_chat_stream).queue()
window_kwargs = {} # or {"server_name": "0.0.0.0", "root_path": "/7860/"}
demo.launch(share=True, debug=True, **window_kwargs) 

## IMPORTANT!! When you're done, please click the Square button (twice to be safe) to stop the session.

<br>

### **예제 2: 내부 응답**

때로는 실제 응답이 사용자에게 나가기 전에 뒤에서 빠른 추론을 수행하고 싶을 때가 있습니다. 이런 작업을 수행하려면 지시를 잘 따르는 사전 성향이 내장된 모델이 필요합니다.

다음은 문장을 몇 개의 클래스 중 하나로 분류하려고 시도하는 "zero-shot 분류" 파이프라인 예시입니다.

**이 zero-shot 분류 체인은 순서대로:**
- `input`과 `options`라는 두 개의 필수 키를 가진 딕셔너리를 받습니다.
- 이를 zero-shot 프롬프트에 통과시켜 LLM에 넣을 입력을 만듭니다.
- 그 문자열을 모델에 전달해 결과를 얻습니다.

**과제:** 이런 종류의 작업에 적합하다고 생각하는 모델을 몇 개 골라 얼마나 잘 수행하는지 확인해 보세요! 구체적으로:
- **여러 예시에 걸쳐 예측 가능한 모델을 찾아보세요.** 형식이 항상 파싱하기 쉽고 매우 예측 가능하다면 그 모델은 아마 괜찮을 것입니다.
- **빠른 모델도 찾아보세요!** 내부 추론은 일반적으로 외부 응답이 생성되기 전에 뒤에서 이루어지기 때문에 중요합니다. 따라서 이는 블로킹 프로세스로, "사용자 대면" 생성의 시작을 지연시켜 시스템이 느리게 느껴지게 만들 수 있습니다.

In [ ]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

## Feel free to try out some more models and see if there are better lightweight options
## https://build.nvidia.com
instruct_llm = ChatNVIDIA(model="nvidia/nemotron-3.5-lightning-30b-a3b", timeout=300, model_kwargs={"chat_template_kwargs": {"enable_thinking": False}})

sys_msg = (
    "Choose the most likely topic classification given the sentence as context."
    " Only one word, no explanation.\n[Options : {options}]"
)

## Zero-shot classification prompt with explicit format assumptions.
zsc_prompt = ChatPromptTemplate.from_messages([
    ("system", sys_msg),
    ("user", "[[{input}]]"),
])

zsc_chain = zsc_prompt | instruct_llm | StrOutputParser()

def zsc_call(input, options=["car", "boat", "airplane", "bike"]):
    return zsc_chain.invoke({"input" : input, "options" : options}).split()[0]

print("-" * 80)
print(zsc_call("Should I take the next exit, or keep going to the next one?"))

print("-" * 80)
print(zsc_call("I get seasick, so I think I'll pass on the trip"))

print("-" * 80)
print(zsc_call("I'm scared of heights, so flying probably isn't for me"))

<br>

### **예제 3: 다중 구성 요소 체인**

이전 예제에서는 딕셔너리를 `prompt -> LLM` 체인에 통과시켜 문자열로 강제 변환하는 방법을 보았으니, 컨테이너 선택의 동기가 되는 쉬운 구조 하나는 확인한 셈입니다. 그렇다면 문자열 출력을 다시 딕셔너리로 변환하는 것도 똑같이 쉬울까요?

**네, 쉽습니다!** 가장 간단한 방법은 사실 LCEL의 *"암묵적 runnable"* 문법을 사용하는 것으로, 함수(체인 포함)의 딕셔너리를 각 함수를 실행하고 그 값을 출력 딕셔너리의 키에 매핑하는 runnable로 사용할 수 있게 해 줍니다.

다음은 이러한 유틸리티를 연습하면서 실전에서 유용할 수 있는 몇 가지 추가 도구도 함께 제공하는 예시입니다.

In [ ]:
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from functools import partial

################################################################################
## Example of dictionary enforcement methods
def make_dictionary(v, key):
    if isinstance(v, dict):
        return v
    return {key : v}

def RInput(key='input'):
    '''Coercing method to mold a value (i.e. string) to in-like dict'''
    return RunnableLambda(partial(make_dictionary, key=key))

def ROutput(key='output'):
    '''Coercing method to mold a value (i.e. string) to out-like dict'''
    return RunnableLambda(partial(make_dictionary, key=key))

def RPrint(preface=""):
    return RunnableLambda(partial(print_and_return, preface=preface))

################################################################################
## Common LCEL utility for pulling values from dictionaries
from operator import itemgetter

up_and_down = (
    RPrint("A: ")
    ## Custom ensure-dictionary process
    | RInput()
    | RPrint("B: ")
    ## Pull-values-from-dictionary utility
    | itemgetter("input")
    | RPrint("C: ")
    ## Anything-in Dictionary-out implicit map
    | {
        'word1' : (lambda x : x.split()[0]),
        'word2' : (lambda x : x.split()[1]),
        'words' : (lambda x: x),  ## <- == to RunnablePassthrough()
    }
    | RPrint("D: ")
    | itemgetter("word1")
    | RPrint("E: ")
    ## Anything-in anything-out lambda application
    | RunnableLambda(lambda x: x.upper())
    | RPrint("F: ")
    ## Custom ensure-dictionary process
    | ROutput()
)

up_and_down.invoke({"input" : "Hello World"})

In [ ]:
## NOTE how the dictionary enforcement methods make it easy to make the following syntax equivalent
up_and_down.invoke("Hello World")

----

<br>

## **Part 4: [실습]** 운율 주제 바꾸기 챗봇

아래는 두 가지 서로 다른 작업을 하나의 에이전트라는 외양 아래 어떻게 구성할 수 있는지 보여 주는 시 생성 예제입니다. 이 시스템은 앞의 간단한 Gradio 예제를 다시 활용하되, 몇 가지 정형화된 응답과 뒤편의 로직을 추가로 확장합니다.

주요 기능은 다음과 같습니다:
- 첫 번째 응답에서는 여러분의 입력을 바탕으로 시를 생성합니다.
- 이후 응답에서는 원래 시의 형식과 구조를 유지하면서 시의 주제를 바꿉니다.

**문제:** 현재 시스템은 첫 번째 부분은 잘 동작하지만, 두 번째 부분은 아직 구현되어 있지 않습니다.

**목표:** 에이전트가 정상적으로 동작하도록 `rhyme_chat2_stream` 메서드의 나머지 부분을 구현하세요.

Gradio 구성 요소를 더 쉽게 다룰 수 있도록, 표준 Python `input` 메서드로 Gradio 채팅 이벤트 루프를 시뮬레이션하는 단순화된 `queue_fake_streaming_gradio` 메서드가 제공됩니다

In [ ]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA

instruct_llm = ChatNVIDIA(model="nvidia/nemotron-3.5-lightning-30b-a3b", timeout=300, model_kwargs={"chat_template_kwargs": {"enable_thinking": False}})
[model.id for model in instruct_llm.available_models]

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from copy import deepcopy

prompt1 = ChatPromptTemplate.from_messages([("user", (
    "INSTRUCTION: Only respond in rhymes"
    "\n\nPROMPT: {input}"
))])

prompt2 =  ChatPromptTemplate.from_messages([("user", (
    "INSTRUCTION: Only responding in rhyme, change the topic of the input poem to be about {topic}!"
    " Make it happy! Try to keep the same sentence structure, but make sure it's easy to recite!"
    " Try not to rhyme a word with itself."
    "\n\nOriginal Poem: {input}"
    "\n\nNew Topic: {topic}"
))])

## These are the main chains, constructed here as modules of functionality.
chain1 = prompt1 | instruct_llm | StrOutputParser()  ## only expects input
chain2 = prompt2 | instruct_llm | StrOutputParser()  ## expects both input and topic

################################################################################
## SUMMARY OF TASK: chain1 currently gets invoked for the first input.
##  Please invoke chain2 for subsequent invocations.

def rhyme_chat2_stream(message, history, return_buffer=True):
    '''This is a generator function, where each call will yield the next entry'''

    first_poem = None
    # Updated to handle the dictionary messaging scheme
    for entry in history:
        if entry.get("role") == "assistant":
            content = entry.get("content", "")
            # Logic to extract the poem from previous assistant response
            if "Let me think!" in content:
                # Splits by the "think" preface and the "rewrite" suffix
                parts = content.split("\n\n")
                if len(parts) > 2:
                    first_poem = parts[1]
                    break

    if first_poem is None:
        ## First Case: Generate the initial poem using chain1
        buffer = "Oh! I can make a wonderful poem about that! Let me think!\n\n"
        yield buffer if return_buffer else buffer

        inst_out = ""
        chat_gen = chain1.stream({"input" : message})
        for token in chat_gen:
            inst_out += token
            buffer += token
            yield buffer if return_buffer else token

        passage = "\n\nNow let me rewrite it with a different focus! What should the new focus be?"
        buffer += passage
        yield buffer if return_buffer else passage

    else:
        ## Subsequent Cases: There is a poem to start with. Generate a similar one with a new topic!

        yield f"Not Implemented!!!"; return ## <- TODO: Comment this out
        
        ########################################################################
        ## TODO: Invoke the second chain to generate the new rhymes.

        # buffer = f"Sure! Here you go!\n\n" ## <- TODO: Uncomment these lines
        # yield buffer
        
        ## TODO: Iterate over stream generator for second generation (using chain2)

        ## END TODO
        ########################################################################

        passage = "\n\nThis is fun! Give me another topic!"
        buffer += passage
        yield buffer if return_buffer else passage

################################################################################
## Below: This is a small-scale simulation of the gradio routine.

def queue_fake_streaming_gradio(chat_stream, history=[], max_questions=3):

    ## Print starter messages
    for entry in history:
        role = entry.get("role").capitalize()
        print(f"\n[ {role} ]:", entry.get("content"))

    ## Mimic the loop
    for _ in range(max_questions):
        user_input = input("\n[ Human ]: ")
        
        # Add user message to history in dict format
        history.append({"role": "user", "content": user_input})
        
        print("\n[ Agent ]: ")
        full_response = ""
        
        # Generator loop
        for token in chat_stream(user_input, history, return_buffer=False):
            print(token, end='')
            full_response += token
            
        # Add assistant message to history in dict format
        history.append({"role": "assistant", "content": full_response})
        print("\n")

## history is of format [[User response 0, Bot response 0], ...]
history = [{"role": "assistant", "content": "Let me help you make a poem! What would you like for me to write?"}]

## Simulating the queueing of a streaming gradio interface, using python input
queue_fake_streaming_gradio(
    chat_stream = rhyme_chat2_stream,
    history = history
)

In [ ]:
## Simple way to initialize history for the ChatInterface
chatbot = gr.Chatbot(value = [{"role": "assistant", "content": "Let me help you make a poem! What would you like for me to write?"}])

## IF USING COLAB: Share=False is faster
gr.ChatInterface(rhyme_chat2_stream, chatbot=chatbot).queue().launch(debug=True, share=True)

----

<br>

## **Part 5: [실습]** 더 깊은 LangChain 통합 사용하기

이 실습은 [**LangServe**](https://github.com/langchain-ai/langserve)와 관련된 예제 코드를 살펴볼 기회를 제공합니다. 구체적으로는 [**`frontend`**](frontend) 디렉터리와 [**`09_langserve.ipynb`**](09_langserve.ipynb) 노트북을 참고합니다.

- [**`09_langserve.ipynb`**](09_langserve.ipynb)로 가서 제공된 스크립트를 실행해 여러 활성 라우트를 가진 서버를 띄우세요.
- 완료되면 아래 **LangServe `RemoteRunnable`** 이 동작하는지 확인하세요. **`RemoteRunnable`** 의 목표는 LangChain 체인을 API 엔드포인트로 쉽게 호스팅할 수 있게 하는 것이므로, 아래는 단지 동작 여부를 확인하는 테스트입니다.
    - 처음에 동작하지 않는다면 실행 순서 문제일 수 있습니다. langserve 노트북을 재시작해 보세요.
 
**이 단계들이 끝나면, 코스의 임의의 노트북에서 다음과 같은 연결이 가능해집니다:**

In [ ]:
from langserve import RemoteRunnable
from langchain_core.output_parsers import StrOutputParser
from langchain_nvidia_ai_endpoints import ChatNVIDIA

llm = RemoteRunnable("http://0.0.0.0:9012/basic_chat/") | StrOutputParser()
for token in llm.stream("Hello World! How is it going?"):
    print(token, end='')

## Equivalent to the following, assuming you're using the same model
# llm = ChatNVIDIA(model="nvidia/nemotron-3.5-lightning-30b-a3b", timeout=300, model_kwargs={"chat_template_kwargs": {"enable_thinking": False}}) | StrOutputParser()
# for token in llm.stream("Hello World! How is it going?"):
#     print(token, end='')

<br>

이 엔드포인트를 활발히 사용하는 곳 중 하나가 `frontend`이며, [**`frontend_server.py`**](././frontend/frontend_server.py) 구현에서 이를 참조합니다:

```python
## Necessary Endpoints
chains_dict = {
    'basic' : RemoteRunnable("http://lab:9012/basic_chat/"),
    'retriever' : RemoteRunnable("http://lab:9012/retriever/"),  ## For the final assessment
    'generator' : RemoteRunnable("http://lab:9012/generator/"),  ## For the final assessment
}

basic_chain = chains_dict['basic']

## Retrieval-Augmented Generation Chain

retrieval_chain = (
    {'input' : (lambda x: x)}
    | RunnableAssign(
        {'context' : itemgetter('input') 
        | chains_dict['retriever'] 
        | LongContextReorder().transform_documents
        | docs2str
    })
)

output_chain = RunnableAssign({"output" : chains_dict['generator'] }) | output_puller
rag_chain = retrieval_chain | output_chain
```

그 결과, '/basic_chat' 체인을 배포하면 프론트엔드 인터페이스의 **"Basic"** 채팅 기능이 구현됩니다. 다시 한번 알려 드리면, 다음 생성된 링크를 통해 프론트엔드에 접근할 수 있습니다: 

<a href="/8090" target="_blank" style="display: inline-block; padding: 12px 24px; background-color: #76b900; color: white; text-decoration: none; border-radius: 4px; font-weight: bold; margin: 3px;">Gradio Frontend UI (<code>/8090</code> -> <code>:8090</code> 안정성을 위해)</a>

In [ ]:
# %%js
# // Manual Access Without NGINX
# var url = 'http://'+window.location.host+':8090';
# element.innerHTML = '<a style="color:green;" target="_blank" href='+url+'><h1>< Link To Gradio Frontend ></h1></a>';

**평가 과제를 시작할 때 이 아이디어를 다시 다루게 됩니다.**

-----
    
**참고:** 이런 유형의 환경에서 LangServe API를 배포하고 의존하는 이 전략은 매우 비표준적이며, 학생들에게 흥미로운 코드를 보여 주기 위해 특별히 만든 것입니다. 최적화된 단일 기능 컨테이너를 사용하면 더 안정적인 구성이 가능하며, [**NVIDIA/GenerativeAIExamples GitHub 저장소**](https://github.com/NVIDIA/GenerativeAIExamples/tree/main/RAG/notebooks)에서 찾아볼 수 있습니다.

-----

<br>

## **Part 6:** 마무리

이 노트북의 목표는 LangChain Expression Language 체계를 익히고, LLM 기능을 제공하기 위한 `gradio`와 `LangServe` 인터페이스를 경험하는 것이었습니다! 다음 노트북에서 이 내용을 더 다루겠지만, 이 노트북은 LLM 에이전트 개발의 중급 및 최신 패러다임으로 나아가는 발판입니다.

### <font color="#76b900">**수고하셨습니다!**</font>

### **다음 단계:**
1. **[선택]** 잠시 시간을 내어 `frontend` 디렉터리의 배포 레시피와 내부 기능을 살펴보세요.
2. **[선택]** 노트북 상단의 **"생각해 볼 질문" 섹션**을 다시 읽고 가능한 답을 생각해 보세요.

---

<div style="width: 55%%; background-color: white; margin-top: 50px;"><center><a href="https://www.nvidia.com/en-us/training/"><img src="https://dli-lms.s3.amazonaws.com/assets/general/nvidia-logo.png" width="300" /></a></center></div>